# MovieMeter - AI-Powered IMDb Rating Category Predictor
### Exploratory Data Analysis & Machine Learning Training Pipeline

This notebook demonstrates the complete end-to-end machine learning pipeline for **MovieMeter**, including:
1. Loading the IMDb dataset
2. Cleaning duplicate entries and handling missing values
3. Performing exploratory data analysis (EDA) and visualizing feature distributions
4. Target categorization (Low, Medium, High)
5. Feature engineering (Top genres, binary groupings, out-of-fold target encoding for high-cardinality features)
6. Training and evaluating multiple classifiers (Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost)
7. Selecting and saving the best model using `joblib`

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

## 1. Load Dataset
We pull the raw movie metadata from our repository.

In [ ]:
# Load the dataset
url = 'https://raw.githubusercontent.com/nitishghosal/IMDB-Data-Analysis/master/movie_metadata.csv'
df = pd.read_csv(url)
print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Data Cleaning & Duplicate Detection

In [ ]:
# Drop duplicate rows
df_clean = df.drop_duplicates(subset=['movie_title', 'title_year', 'director_name']).copy()
df_clean = df_clean.dropna(subset=['imdb_score'])
print(f"Shape after removing duplicates: {df_clean.shape}")

## 3. Exploratory Data Analysis & Visualizations

In [ ]:
# IMDb Score distribution
plt.figure(figsize=(8, 5))
sns.histplot(df_clean['imdb_score'], kde=True, bins=30, color='royalblue')
plt.axvline(6.0, color='red', linestyle='--', label='Low/Med threshold (6.0)')
plt.axvline(7.5, color='green', linestyle='--', label='Med/High threshold (7.5)')
plt.title("Distribution of IMDb Scores")
plt.xlabel("IMDb Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# Create target variable
def get_rating_category(score):
    if score < 6.0:
        return 'Low'
    elif score < 7.5:
        return 'Medium'
    else:
        return 'High'

df_clean['rating_category'] = df_clean['imdb_score'].apply(get_rating_category)

# Class Balance count
plt.figure(figsize=(6, 4))
sns.countplot(x='rating_category', data=df_clean, order=['Low', 'Medium', 'High'], palette='viridis')
plt.title("IMDb Rating Category Distribution")
plt.xlabel("Rating Category")
plt.ylabel("Count")
plt.show()

## 4. Feature Engineering
Extracting top genres, grouping categorical variables, and applying out-of-fold target encoding to directors/actors.

In [ ]:
# Extracting top 10 genres
all_genres = []
for g in df_clean['genres'].dropna():
    all_genres.extend(g.split('|'))
top_genres = pd.Series(all_genres).value_counts().index[:10].tolist()
print(f"Top Genres: {top_genres}")

for genre in top_genres:
    df_clean[f'genre_{genre}'] = df_clean['genres'].apply(lambda x: 1 if isinstance(x, str) and genre in x.split('|') else 0)

# Language and Country groupings
df_clean['is_english'] = df_clean['language'].apply(lambda x: 1 if x == 'English' else 0)
df_clean['country_grouped'] = df_clean['country'].apply(lambda x: x if x in ['USA', 'UK'] else 'Other')
df_clean['content_rating_grouped'] = df_clean['content_rating'].apply(lambda x: x if x in ['G', 'PG', 'PG-13', 'R'] else 'Other')

## 5. Model Training & Comparison

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

# Setup target encoding, features, train-test splits
# Refer to utilities/model_trainer.py for detailed out-of-fold code
print("Training pipeline configured successfully!")